In [ ]:
from __future__ import annotations

import os
import csv
import pickle
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
from Bio import SeqIO

In [ ]:
# ============================================================
# Project / paths
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()
PROJECT_ROOT = REPOSITORY_ROOT / "NT"
DATA_ROOT = PROJECT_ROOT / "data"

SIM_DIR = DATA_ROOT / "sim"
EVO_DISTANCE_DIR = DATA_ROOT / "evo_distances"

MANIFEST_PATH = DATA_ROOT / "manifests" / "simulation_manifest.csv"


# ============================================================
# Evolutionary-distance parameters
# ============================================================

STANDARD_NT = "ACGT"

EPS = 1e-12

# JC69:
# d = -(3/4) log(1 - (4/3)p)
JC69_MAX_P = 3.0 / 4.0

# False: keep existing evolutionary-distance outputs
# True: regenerate existing outputs
OVERWRITE = False

In [ ]:
@dataclass
class EvoDistanceRow:
    tag: str
    fasta_path: str
    n_sequences: int

    pickle_path: str
    npz_path: str

    status: str
    message: str


def ensure_directories() -> None:
    EVO_DISTANCE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


def check_inputs() -> None:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Simulation manifest was not found: {MANIFEST_PATH}"
        )


def read_simulation_manifest() -> list[dict]:
    rows: list[dict] = []

    with MANIFEST_PATH.open(newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("status", "") in {"ok", "skipped_existing"}:
                rows.append(row)

    return rows


def resolve_fasta_path(
    row: dict,
) -> Path:
    manifest_path = Path(row["fasta_path"]).expanduser()

    if manifest_path.is_absolute():
        candidates = [manifest_path]
    else:
        candidates = [PROJECT_ROOT / manifest_path]

    # Backward-compatible fallback for manifests that only encode the tag.
    candidates.append(
        SIM_DIR / f"{row['tag']}.fa"
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"FASTA file was not found for tag={row['tag']}. "
        f"Checked: {checked}"
    )


def read_fasta_records(
    fasta_path: Path,
) -> tuple[list[str], list[str]]:
    records = list(
        SeqIO.parse(
            str(fasta_path),
            "fasta",
        )
    )

    ids = [
        record.id
        for record in records
    ]

    sequences = [
        str(record.seq).upper()
        for record in records
    ]

    return ids, sequences


def encode_nt_sequences(
    sequences: list[str],
) -> np.ndarray:
    nt_to_index = {
        nt: index
        for index, nt in enumerate(STANDARD_NT)
    }

    n = len(sequences)
    length = len(sequences[0])

    encoded = np.full(
        (n, length),
        fill_value=-1,
        dtype=int,
    )

    for i, sequence in enumerate(sequences):
        if len(sequence) != length:
            raise ValueError(
                "All sequences must have the same length."
            )

        for j, nt in enumerate(sequence):
            encoded[i, j] = nt_to_index.get(
                nt,
                -1,
            )

    return encoded


def pairwise_p_distance_nt(
    encoded: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    n = encoded.shape[0]

    p_distance = np.zeros(
        (n, n),
        dtype=float,
    )

    valid_sites = np.zeros(
        (n, n),
        dtype=int,
    )

    for i in range(n):
        for j in range(i + 1, n):
            valid = (
                (encoded[i] >= 0)
                & (encoded[j] >= 0)
            )

            n_valid = int(
                valid.sum()
            )

            valid_sites[i, j] = n_valid
            valid_sites[j, i] = n_valid

            if n_valid == 0:
                p = np.nan

            else:
                mismatches = (
                    encoded[i, valid]
                    != encoded[j, valid]
                )

                p = float(
                    mismatches.mean()
                )

            p_distance[i, j] = p
            p_distance[j, i] = p

    np.fill_diagonal(
        p_distance,
        0.0,
    )

    return p_distance, valid_sites


def jc69_corrected_distance(
    p_distance: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    p = np.asarray(
        p_distance,
        dtype=float,
    )

    clip_threshold = JC69_MAX_P - EPS

    clipped_p = np.clip(
        p,
        0.0,
        clip_threshold,
    )

    clipped_mask = (
        p >= JC69_MAX_P
    )

    distance = -(
        3.0
        / 4.0
    ) * np.log(
        1.0
        - (
            4.0
            / 3.0
        )
        * clipped_p
    )

    finite = distance[np.isfinite(distance)]

    fallback = (
        float(finite.max())
        if finite.size > 0
        else 0.0
    )

    distance = np.nan_to_num(
        distance,
        nan=fallback,
        posinf=fallback,
        neginf=0.0,
    )

    distance = 0.5 * (
        distance
        + distance.T
    )

    np.fill_diagonal(
        distance,
        0.0,
    )

    return distance, clipped_mask


def output_is_complete(
    pickle_path: Path,
    npz_path: Path,
) -> bool:
    if not pickle_path.exists():
        return False

    if not npz_path.exists():
        return False

    try:
        with pickle_path.open("rb") as f:
            payload = pickle.load(f)

    except Exception:
        return False

    required_keys = {
        "p_distance",
        "valid_sites",
        "jc69_distance",
    }

    return required_keys.issubset(
        payload.keys()
    )


def save_pickle(
    obj,
    output_path: Path,
) -> None:
    with output_path.open("wb") as f:
        pickle.dump(
            obj,
            f,
        )


def save_npz(
    output_path: Path,
    **arrays,
) -> None:
    np.savez_compressed(
        output_path,
        **arrays,
    )


def save_evo_manifest(
    rows: list[EvoDistanceRow],
    output_path: Path,
) -> None:
    fieldnames = list(
        EvoDistanceRow.__annotations__.keys()
    )
    path_fields = {
        "fasta_path",
        "pickle_path",
        "npz_path",
    }
    root = PROJECT_ROOT.resolve()

    with output_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:
            record = asdict(row)

            for field in path_fields:
                value = record.get(field)

                if not value:
                    continue

                path = Path(value).expanduser().resolve()

                try:
                    record[field] = (
                        path.relative_to(root).as_posix()
                    )
                except ValueError as error:
                    raise ValueError(
                        f"{field} is outside project root: {path}"
                    ) from error

            writer.writerow(record)


def process_one_dataset(
    tag: str,
    fasta_path: Path,
) -> EvoDistanceRow:
    pickle_path = (
        EVO_DISTANCE_DIR
        / f"{tag}_evo_distances.pkl"
    )

    npz_path = (
        EVO_DISTANCE_DIR
        / f"{tag}_evo_distances.npz"
    )

    row = EvoDistanceRow(
        tag=tag,
        fasta_path=str(fasta_path),
        n_sequences=0,
        pickle_path=str(pickle_path),
        npz_path=str(npz_path),
        status="planned",
        message="",
    )

    if (
        output_is_complete(
            pickle_path,
            npz_path,
        )
        and not OVERWRITE
    ):
        return replace(
            row,
            status="ok",
            message="skipped_existing",
        )

    try:
        ids, sequences = read_fasta_records(
            fasta_path
        )

        encoded = encode_nt_sequences(
            sequences
        )

        p_distance, valid_sites = pairwise_p_distance_nt(
            encoded
        )

        jc69_distance, jc69_clipped = jc69_corrected_distance(
            p_distance
        )

        payload = {
            "tag": tag,
            "ids": ids,
            "p_distance": p_distance,
            "valid_sites": valid_sites,
            "jc69_distance": jc69_distance,
            "jc69_clipped": jc69_clipped,
        }

        save_pickle(
            payload,
            pickle_path,
        )

        save_npz(
            npz_path,
            p_distance=p_distance,
            valid_sites=valid_sites,
            jc69_distance=jc69_distance,
            jc69_clipped=jc69_clipped,
        )

    except Exception as error:
        return replace(
            row,
            status="failed",
            message=str(error)[:2000],
        )

    return replace(
        row,
        n_sequences=len(ids),
        status="ok",
        message="",
    )

In [ ]:
ensure_directories()
check_inputs()

simulation_rows = read_simulation_manifest()

evo_rows: list[EvoDistanceRow] = []

for index, sim_row in enumerate(simulation_rows, start=1):
    tag = sim_row["tag"]
    fasta_path = resolve_fasta_path(
        sim_row
    )

    print(
        f"[{index:04d}/{len(simulation_rows):04d}] "
        f"{tag}"
    )

    evo_rows.append(
        process_one_dataset(
            tag=tag,
            fasta_path=fasta_path,
        )
    )

evo_manifest_path = (
    EVO_DISTANCE_DIR
    / "evo_distance_manifest.csv"
)

save_evo_manifest(
    evo_rows,
    evo_manifest_path,
)

status_counts = Counter(
    row.status
    for row in evo_rows
)

message_counts = Counter(
    row.message
    for row in evo_rows
)

print()
print("Evolutionary distance construction completed.")

for status, count in sorted(status_counts.items()):
    print(f"{status}: {count}")

print()
print("Messages:")

for message, count in sorted(message_counts.items()):
    label = message if message else "generated"
    print(f"{label}: {count}")

print()
print(f"Evolutionary distance manifest: {evo_manifest_path}")